# SQL or Custom Expressions?

SLayer draws a clean line between **SQL** and its **DSL**:

- **Model definitions** (dimensions, measures, filters) use **raw SQL** — they reference the underlying table columns directly.
- **Queries** use the **DSL** — they reference dimension/measure names, apply transforms like `cumsum` and `change`, and filter on named entities.

The model is the abstraction boundary between the two. This notebook demonstrates the distinction with working examples.

See also: [SQL vs DSL](sql_vs_dsl.md) | [Formulas](../../concepts/formulas.md) | [Models](../../concepts/models.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

/home/james/miniconda3/envs/motley3.11/lib/python3.11/site-packages/sqlglot/tokens.py:14: UserWarning: sqlglot[rs] is deprecated and no longer compatible with sqlglot. Please use sqlglotc instead for faster parsing: pip install sqlglot[c]
  warnings.warn(


## Model Definitions Use SQL

A model's dimensions and measures define the mapping from semantic names to SQL expressions. The `sql` field is a raw SQL column reference or expression — it's what ends up in the generated `SELECT` and `GROUP BY`.

For example, the `orders` model maps the dimension name `order_total` to the SQL column `order_total`, and the measure name `order_total_sum` to `SUM(order_total)`:

In [2]:
orders_model = next(m for m in models if m.name == "orders")

print("Dimensions (name -> SQL expression):")
for dim in orders_model.dimensions[:7]:  # own columns only
    print(f"  {dim.name:<20} -> sql: {dim.sql!r:<20} type: {dim.type}")

print("\nMeasures (name -> SQL expression + aggregation):")
for m in orders_model.measures[:6]:
    sql_str = m.sql or '(*)'
    print(f"  {m.name:<20} -> sql: {sql_str!r:<20} type: {m.type}")

Dimensions (name -> SQL expression):
  id                   -> sql: 'id'                 type: string
  customer_id          -> sql: 'customer_id'        type: string
  ordered_at           -> sql: 'ordered_at'         type: date
  store_id             -> sql: 'store_id'           type: string
  subtotal             -> sql: 'subtotal'           type: number
  tax_paid             -> sql: 'tax_paid'           type: number
  order_total          -> sql: 'order_total'        type: number

Measures (name -> SQL expression + aggregation):
  count                -> sql: '(*)'                type: count
  subtotal_sum         -> sql: 'subtotal'           type: sum
  subtotal_avg         -> sql: 'subtotal'           type: avg
  subtotal_min         -> sql: 'subtotal'           type: min
  subtotal_max         -> sql: 'subtotal'           type: max
  subtotal_distinct    -> sql: 'subtotal'           type: count_distinct


Notice that the dimension/measure `name` is a semantic label, while `sql` is the raw SQL expression. They often coincide (e.g., `name: "order_total"`, `sql: "order_total"`), but they don't have to — a dimension could be named `revenue` with `sql: "amount * quantity"`.

Model-level filters are also raw SQL:

```yaml
filters:
  - "deleted_at IS NULL"
  - "status <> 'test'"
```

## Queries Use the DSL

When you write a query, you reference **dimension and measure names** — the semantic side. You never write SQL in a query.

Transform functions like `cumsum`, `change`, and `time_shift` are part of the DSL. They compile to complex SQL (CTEs, window functions, self-joins), but you just name them:

In [3]:
result = engine.execute(query={
    "source_model": "orders",
    "time_dimensions": [{"dimension": {"name": "ordered_at"}, "granularity": "month"}],
    "fields": [
        {"formula": "order_total_sum"},
        {"formula": "change(order_total_sum)", "name": "mom_change"},
    ],
    "order": [{"column": {"name": "ordered_at"}, "direction": "asc"}],
    "limit": 6,
})

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12}")
print("-" * 38)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12}")

Month             Revenue   MoM Change
--------------------------------------
2018-09      $   4,738.80            -
2018-10      $   7,882.42      $+3,144
2018-11      $  11,394.11      $+3,512
2018-12      $  14,904.61      $+3,511
2019-01      $  17,790.62      $+2,886
2019-02      $  17,464.28        $-326


The `change()` function compiled to a self-join CTE under the hood — but the query only referenced the measure name `order_total_sum` and the transform name `change`. No SQL was written.

## Filters in Queries Reference Model Entities

Query-level filters use dimension and measure names — including dotted names for joined dimensions. SLayer resolves them to the correct SQL automatically:

In [4]:
# Filter on a joined dimension name — not a SQL column reference
result = engine.execute(query={
    "source_model": "orders",
    "fields": [{"formula": "count"}, {"formula": "order_total_sum"}],
    "dimensions": [{"name": "stores.name"}],
    "filters": ["stores.name in ('Brooklyn', 'Philadelphia')"],
})

for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders.count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

print(f"\nThe filter 'stores.name' resolved to SQL:")
# Show just the WHERE clause from the generated SQL
where_line = [line for line in result.sql.split('\n') if 'WHERE' in line or 'IN' in line]
for line in where_line:
    print(f"  {line.strip()}")

  Brooklyn: 80,349 orders, $1,014,796.82
  Philadelphia: 58,293 orders, $753,880.60

The filter 'stores.name' resolved to SQL:
  LEFT JOIN stores AS stores ON orders.store_id = stores.id
  WHERE
  stores.name IN ('Brooklyn', 'Philadelphia')


## Adding SQL-Level Filters via ModelExtension

What if you need a filter that references raw SQL — an expression over underlying columns that isn't a named dimension?

Use `ModelExtension` to add the filter to the **model definition** inline, right inside the query. Model-level filters are SQL, so any valid SQL expression works:

In [5]:
# "subtotal > tax_paid * 5" is raw SQL — it references underlying table columns,
# not dimension names. So it goes on the model (via ModelExtension), not the query.
result = engine.execute(query={
    "source_model": {
        "source_name": "orders",
        "filters": ["subtotal > tax_paid * 5"],
    },
    "fields": [{"formula": "count"}, {"formula": "order_total_sum"}],
    "dimensions": [{"name": "stores.name"}],
    "order": [{"column": {"name": "order_total_sum"}, "direction": "desc"}],
})

print("Orders where subtotal > 5x tax (raw SQL filter via ModelExtension):")
for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders.count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

Orders where subtotal > 5x tax (raw SQL filter via ModelExtension):
  Brooklyn: 80,349 orders, $1,014,796.82
  Philadelphia: 58,293 orders, $753,880.60
  Chicago: 34,997 orders, $452,218.31
  San Francisco: 28,453 orders, $373,174.21
  New Orleans: 4,293 orders, $55,509.93


The key insight: the raw SQL filter `subtotal > tax_paid * 5` is on the **model** (via `ModelExtension`), while the query's own `fields`, `dimensions`, and any query-level `filters` stay in DSL territory.

## Query Result as Model

What if you want to use DSL transforms like `time_shift` or `change` to define derived measures or dimensions?

Use `create_model_from_query()` to save a query's result as a permanent model. The derived columns become dimensions and measures on the new model, which can then be queried like any other.

See [Creating Models from Queries](../../concepts/models.md#creating-models-from-queries) for details.

## Summary

| Layer | Language | Examples |
|-------|----------|----------|
| **Model definitions** (dimensions, measures, filters) | Raw SQL | `sql: "amount"`, `filters: ["deleted_at IS NULL"]` |
| **Queries** (fields, dimensions, filters) | DSL names | `{"formula": "change(revenue)"}`, `filters: ["status = 'active'"]` |
| **SQL in queries** | ModelExtension | Add raw SQL filters/dimensions to the model inline |
| **DSL in models** | Query-as-model | Save a DSL query result as a permanent model |